# Does POD represent triple-point motion?

This diagnostic reads your completed POD coefficients and reviewed TP positions. It measures training-only mode–TP relationships and tests continuous TP-position prediction on **validation**, without using final test outcomes. It compares leading modes, training-ranked modes, and optionally a new crop.

The first four modes need not be the modes most related to TP motion. Weak individual correlations do not exclude information distributed across modes or nonlinear relationships. This notebook is a diagnostic, not a new final test or a proof of causation.

## 1. Parameters

Place this notebook in your Hemisphere-QML project. Existing inputs are read only. Each run creates a fresh diagnostic output folder. No raw images are required unless you enable the optional crop study.

The default paths match your `results-TP` experiment. Change `RESULTS_DIRECTORY` if your model outputs are elsewhere. Set `CROP_STUDIES` only after inspecting where the full TP excursion lies; no tighter crop is assumed to be appropriate automatically.

In [ ]:
%matplotlib inline
from pathlib import Path
from datetime import datetime
import csv, json, hashlib, warnings
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, balanced_accuracy_score
from IPython.display import display

PROJECT_DIR = Path.cwd().resolve()
TP_DIRECTORY = PROJECT_DIR / "results-TP" / "tp_measurement"
RESULTS_DIRECTORY = TP_DIRECTORY / "categories_pod_qsvm"
STATES_CSV = TP_DIRECTORY / "categories" / "tp_states.csv"
CATEGORY_DEFINITION = TP_DIRECTORY / "categories" / "tp_category_definition.json"
OUTPUT_ROOT = PROJECT_DIR / "results-TP" / "pod_tp_diagnostics"

MODE_COUNTS = [2, 4, 8, 12, 16, 24]
RIDGE_ALPHAS = [0.01, 0.1, 1.0, 10.0, 100.0]
RUN_NONLINEAR = True
SVR_C_VALUES = [10.0, 100.0]  # Target units are original sensor pixels.
SVR_EPSILON_PX = 0.5
MAX_REGRESSION_TRAIN = 2000  # Identical deterministic training rows for both regressors.
FIGURE_DPI = 300

RUN_CROP_STUDY = False
CROP_STUDIES = {}  # Example syntax: {"tighter_tp": [x0, y0, x1, y1]}
CROP_BIN_FACTOR = 2
CROP_POD_RANK = 24
KEEP_CROP_CACHE = False

plt.rcParams.update({"font.size":11,"axes.spines.top":False,
                     "axes.spines.right":False,"savefig.bbox":"tight"})

def read_csv(path):
    with Path(path).open(newline="", encoding="utf-8-sig") as f:
        reader = csv.DictReader(f)
        if not reader.fieldnames or len(reader.fieldnames) != len(set(reader.fieldnames)):
            raise ValueError(f"Missing/duplicate CSV columns: {path}")
        rows = list(reader)
    if any(None in row for row in rows):
        raise ValueError(f"Malformed CSV rows: {path}")
    return rows

def write_csv(path, rows):
    if not rows: return
    with Path(path).open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0]))
        writer.writeheader(); writer.writerows(rows)

def digest(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def save_figure(fig, name):
    for ext in ("png", "pdf", "svg"):
        fig.savefig(OUTPUT_DIRECTORY / f"{name}.{ext}", dpi=FIGURE_DPI)
    display(fig); plt.close(fig)

## 2. Align measurements and use only training/validation rows

Filenames, camera indices, timing, partitions, reviewed status, and saved TP definitions are checked. Numeric test coefficients and TP coordinates are never used. The training y mean/SD must agree with the saved category definition. A saved definition in the model directory must match the current measurement experiment.

In [ ]:
coefficient_path = RESULTS_DIRECTORY / "modal_coefficients.csv"
config_path = RESULTS_DIRECTORY / "config_used.json"
cfg = json.loads(config_path.read_text())
definition = json.loads(CATEGORY_DEFINITION.read_text())
if definition.get("fit_partition") != "train":
    raise ValueError("Expected training-defined TP categories.")
saved_definition_path = RESULTS_DIRECTORY / "tp_category_definition.json"
if saved_definition_path.is_file():
    saved_definition = json.loads(saved_definition_path.read_text())
    if saved_definition.get("tracks_sha256") != definition.get("tracks_sha256"):
        raise ValueError("TP states and model outputs use different tracking experiments.")
all_coefficients = read_csv(coefficient_path)
all_states = read_csv(STATES_CSV)
lookup = {r["filename"]: r for r in all_states}
if len(lookup) != len(all_states) or len({r["filename"] for r in all_coefficients}) != len(all_coefficients):
    raise ValueError("Duplicate filenames in input tables.")
if set(lookup) != {r["filename"] for r in all_coefficients}:
    raise ValueError("TP states and POD coefficients have different image filenames.")
mode_columns = sorted([k for k in all_coefficients[0] if k.startswith("a_") and k[2:].isdigit()],
                      key=lambda k:int(k[2:]))
if not mode_columns: raise ValueError("No a_1, a_2, ... coefficients found.")
selected = [r for r in all_coefficients if r["partition"] in ("train", "validation")]
if not selected: raise ValueError("No saved training/validation rows found.")
for r in selected:
    tp = lookup[r["filename"]]
    if int(r["sequence_frame_index"]) != int(tp["sequence_frame_index"]):
        raise ValueError("Camera frame indices do not match.")
    if not np.isclose(float(r["time_s"]), float(tp["time_s"]), rtol=0, atol=1e-10):
        raise ValueError("TP and POD timestamps differ.")
    if tp["accepted"] != "True": raise ValueError("Unresolved TP tracks remain.")
X = np.asarray([[float(r[k]) for k in mode_columns] for r in selected])
y = np.asarray([float(lookup[r["filename"]]["y_px"]) for r in selected])
time_ms = (np.asarray([float(r["time_s"]) for r in selected]) - float(selected[0]["time_s"])) * 1000
train = np.flatnonzero([r["partition"] == "train" for r in selected])
validation = np.flatnonzero([r["partition"] == "validation" for r in selected])
if len(train) < 3 or len(validation) < 2: raise ValueError("Need at least 3 training and 2 validation rows.")
if not np.isfinite(X).all() or not np.isfinite(y).all(): raise ValueError("Nonfinite training/validation data.")
if np.std(y[train]) == 0: raise ValueError("Training TP y has zero spread.")
if len(train) != definition["training_frames"] or not np.allclose(
    [y[train].mean(),y[train].std(ddof=0)],
    [definition["mean_y_px"],definition["standard_deviation_y_px"]],rtol=1e-8,atol=1e-8):
    raise ValueError("Saved category definition does not match training TP positions.")

OUTPUT_DIRECTORY = OUTPUT_ROOT / datetime.now().strftime("run_%Y%m%d_%H%M%S_%f")
for protected in (RESULTS_DIRECTORY, TP_DIRECTORY / "categories", Path(cfg["image_directory"])):
    protected = protected.expanduser().resolve()
    if OUTPUT_DIRECTORY.resolve() == protected or protected in OUTPUT_DIRECTORY.resolve().parents:
        raise ValueError("Diagnostic output must be separate from inputs.")
OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=False)
print(f"Training rows: {len(train):,}; validation rows: {len(validation):,}; available modes: {X.shape[1]}")
print("Final test/embargo rows excluded from all numeric analysis.")
print("Outputs:", OUTPUT_DIRECTORY)

## 3. Which individual modes relate to TP position?

Pearson captures linear association; Spearman captures monotonic association. Both are descriptive training correlations. Autocorrelated frames do not provide independent samples for ordinary significance tests, so no naive p-values are reported. Constant modes are marked missing. POD mode signs are arbitrary; correlation magnitude matters more than sign.

In [ ]:
def training_correlations(features):
    pearson, spearman = [], []
    for j in range(features.shape[1]):
        column = features[train,j]
        if np.std(column) == 0:
            pearson.append(np.nan); spearman.append(np.nan)
        else:
            pearson.append(float(np.corrcoef(column,y[train])[0,1]))
            spearman.append(float(spearmanr(column,y[train]).statistic))
    return np.asarray(pearson), np.asarray(spearman)

pearson, spearman = training_correlations(X)
correlation_rows = [dict(mode=j+1,pearson=pearson[j],spearman=spearman[j]) for j in range(X.shape[1])]
write_csv(OUTPUT_DIRECTORY / "training_mode_tp_correlations.csv", correlation_rows)
ranked_modes = np.argsort(-np.nan_to_num(np.abs(pearson),nan=-1), kind="stable")
print("Modes ranked by absolute training Pearson correlation:", (ranked_modes+1).tolist())
fig, ax = plt.subplots(figsize=(10,4), layout="constrained")
positions = np.arange(1,X.shape[1]+1)
ax.bar(positions-.18,pearson,width=.36,label="Pearson",color="#0072B2")
ax.bar(positions+.18,spearman,width=.36,label="Spearman",color="#D55E00")
ax.axhline(0,color="0.4",lw=.7); ax.set_ylim(-1,1)
ax.set(xlabel="POD mode",ylabel="Training correlation with TP y",
       title="Individual mode association with vertical TP position")
ax.legend(frameon=False)
save_figure(fig,"01_training_correlations")

### Compare time histories and scatter plots

These plots show the four modes with strongest absolute **training** Pearson correlation. Standardization is for the time-history display only. Scatter plots can reveal curved relationships that a single correlation misses.

In [ ]:
shown = ranked_modes[:min(4,X.shape[1])]
fig, axes = plt.subplots(len(shown),1,figsize=(10,2.2*len(shown)),sharex=True,layout="constrained",squeeze=False)
z_y = (y[train]-y[train].mean())/y[train].std()
for ax,j in zip(axes.flat,shown):
    column = X[train,j]
    z_a = (column-column.mean())/column.std() if column.std()>0 else np.zeros(len(column))
    ax.plot(time_ms[train],z_y,color="black",lw=.9,label="TP y")
    ax.plot(time_ms[train],z_a,color="#0072B2",lw=.7,label=f"Mode {j+1}")
    ax.set_ylabel("Standardized value"); ax.legend(frameon=False,loc="upper right")
axes[-1,0].set_xlabel("Time from first selected frame (ms)")
save_figure(fig,"02_training_time_histories")
fig,axes=plt.subplots(1,len(shown),figsize=(3.5*len(shown),3.5),layout="constrained",squeeze=False)
for ax,j in zip(axes.flat,shown):
    column = X[train,j]
    z_a = (column-column.mean())/column.std() if column.std()>0 else np.zeros(len(column))
    ax.hexbin(z_a,y[train],gridsize=35,mincnt=1,cmap="viridis")
    ax.set(xlabel=f"Standardized mode {j+1}",ylabel="TP y (sensor px)",title=f"Training r={pearson[j]:.2f}")
    ax.invert_yaxis()
save_figure(fig,"03_training_tp_scatter")

## 4. Can combinations of modes predict continuous TP position?

Compare leading modes with modes ranked using training data only. Ridge is a simple linear regression; optional RBF-SVR probes nonlinear relationships. Scalers and models fit training samples only. Hyperparameters and feature count are compared on validation RMSE; this validation-selected score is optimistic relative to a fresh final test.

The baseline always predicts the training-mean TP y. All regressors use the same deterministic capped training rows and all validation rows. No time coordinate or TP coordinate is a predictor.

In [ ]:
if isinstance(MAX_REGRESSION_TRAIN,bool) or not isinstance(MAX_REGRESSION_TRAIN,int) or MAX_REGRESSION_TRAIN < 3:
    raise ValueError("MAX_REGRESSION_TRAIN must be an integer >=3.")
fit_rows = train[np.linspace(0,len(train)-1,min(MAX_REGRESSION_TRAIN,len(train)),dtype=int)]
counts = sorted(set(k for k in MODE_COUNTS if isinstance(k,int) and not isinstance(k,bool) and 1<=k<=X.shape[1]))
if not counts: raise ValueError("No requested feature count is available in saved POD.")
upper = float(definition["upper_boundary_y_px"])
lower = float(definition["lower_boundary_y_px"])
def categories(values):
    return np.where(values>lower,"candidate_1",np.where(values<upper,"candidate_3","candidate_2"))

def regression_metrics(predicted):
    return dict(mae_px=float(mean_absolute_error(y[validation],predicted)),
                rmse_px=float(np.sqrt(mean_squared_error(y[validation],predicted))),
                r2=float(r2_score(y[validation],predicted)),
                category_accuracy=float(np.mean(categories(y[validation])==categories(predicted))))

baseline_prediction = np.full(len(validation),y[train].mean())
baseline = regression_metrics(baseline_prediction)
print("Training-mean baseline (validation):",baseline)
search_rows=[]; winners={}
def evaluate_features(features, experiment):
    correlations,_ = training_correlations(features)
    ranking = np.argsort(-np.nan_to_num(np.abs(correlations),nan=-1),kind="stable")
    available_counts = sorted(set(k for k in MODE_COUNTS
        if isinstance(k,int) and not isinstance(k,bool) and 1<=k<=features.shape[1]))
    if not available_counts: raise ValueError("No requested mode count fits this POD rank.")
    for ordering in ("leading","TP_ranked"):
        for k in available_counts:
            columns = np.arange(k) if ordering=="leading" else ranking[:k]
            candidates=[("Ridge",dict(alpha=alpha)) for alpha in RIDGE_ALPHAS]
            if RUN_NONLINEAR:
                candidates += [("RBF_SVR",dict(C=c,epsilon=SVR_EPSILON_PX,gamma="scale")) for c in SVR_C_VALUES]
            for family,parameters in candidates:
                estimator = Ridge(**parameters) if family=="Ridge" else SVR(**parameters)
                model = make_pipeline(StandardScaler(),estimator)
                model.fit(features[fit_rows][:,columns],y[fit_rows])
                predicted = model.predict(features[validation][:,columns])
                result=dict(experiment=experiment,ordering=ordering,model=family,n_modes=k,
                            modes=";".join(str(j+1) for j in columns),parameters=json.dumps(parameters),
                            n_fit=len(fit_rows),n_validation=len(validation),**regression_metrics(predicted))
                search_rows.append(result)
                previous=winners.get(experiment)
                if previous is None or result["rmse_px"]<previous["metrics"]["rmse_px"]:
                    winners[experiment]=dict(metrics=result,prediction=predicted.copy())
    print(experiment,"best validation result:",winners[experiment]["metrics"])

evaluate_features(X,"saved_POD")

## 5. Optional tighter-crop sensitivity study

To compare image regions, set `RUN_CROP_STUDY=True` and supply one or more `CROP_STUDIES` in original sensor pixels. Ensure every crop contains the TP excursion and interacting shocks. A crop excluding TP may test indirect flow information, but should be labeled as such.

Only training and validation TIFFs are read. A disk-backed image matrix bounds memory; POD fits training images only. This can use substantial disk/time. Raw TIFFs are never changed, and the existing POD is preserved. Crop results use the same frame splits and regression rows. This is a validation sensitivity analysis, not another final test.

In [ ]:
if RUN_CROP_STUDY:
    if not CROP_STUDIES: raise ValueError("Set CROP_STUDIES before enabling the study.")
    from schlieren_qml.triple_point import _sequence
    from schlieren_qml.data import read_image, preprocess_image
    from schlieren_qml.pod import fit_pod
    paths,_ = _sequence(cfg)
    image_lookup = {p.name:p for p in paths}
    for number,(name,roi) in enumerate(CROP_STUDIES.items(),start=1):
        crop_folder=OUTPUT_DIRECTORY/f"crop_{number:02d}"
        crop_folder.mkdir()
        first=preprocess_image(read_image(image_lookup[selected[0]["filename"]]),roi,CROP_BIN_FACTOR)
        if not 1<=CROP_POD_RANK<=min(len(train)-1,first.size):
            raise ValueError("CROP_POD_RANK exceeds training/pixel dimensions.")
        cache_path=crop_folder/"train_validation_images.npy"
        cache=np.lib.format.open_memmap(cache_path,mode="w+",dtype="float32",shape=(len(selected),first.size))
        try:
            for i,row in enumerate(selected):
                frame=preprocess_image(read_image(image_lookup[row["filename"]]),roi,CROP_BIN_FACTOR)
                if frame.shape != first.shape: raise ValueError("Image geometry changed.")
                cache[i]=frame.ravel()
                if (i+1)%1000==0: print(name,f"{i+1:,}/{len(selected):,}",flush=True)
            cache.flush()
            model=fit_pod(cache,train,rank=CROP_POD_RANK,seed=42)
            features=model.transform(cache)
            model.save(crop_folder/"pod_model.npz")
            np.save(crop_folder/"modal_coefficients.npy",features)
            (crop_folder/"crop_definition.json").write_text(json.dumps(dict(
                name=name,roi=roi,bin_factor=CROP_BIN_FACTOR,rank=CROP_POD_RANK,
                filenames=[r["filename"] for r in selected],fit_partition="train"),indent=2))
        finally:
            cache.flush();cache._mmap.close();del cache
            if not KEEP_CROP_CACHE: cache_path.unlink(missing_ok=True)
        evaluate_features(features,f"crop_{number:02d}: {name}")
else:
    print("Crop study disabled; saved-POD diagnostics require no TIFF reads.")

## 6. Compare validation errors and inspect position predictions

Lower RMSE/MAE indicates better continuous-position prediction. R² ≤ 0 means performance no better than predicting the validation mean (a different baseline from the saved training mean). Category accuracy here comes from thresholding regression predictions; it is not your SVM/QSVM accuracy. Review errors near either boundary.

In [ ]:
write_csv(OUTPUT_DIRECTORY/"validation_regression_search.csv",search_rows)
write_csv(OUTPUT_DIRECTORY/"validation_winners.csv",[v["metrics"] for v in winners.values()])
fig,ax=plt.subplots(figsize=(10,4.5),layout="constrained")
styles={"Ridge":"o-","RBF_SVR":"s--"}
for experiment in winners:
    for ordering in ("leading","TP_ranked"):
        for family in ("Ridge","RBF_SVR"):
            rows=[r for r in search_rows if r["experiment"]==experiment and r["ordering"]==ordering and r["model"]==family]
            if not rows:continue
            ks=sorted(set(r["n_modes"] for r in rows))
            errors=[min(r["rmse_px"] for r in rows if r["n_modes"]==k) for k in ks]
            ax.plot(ks,errors,styles[family],label=f"{experiment}: {ordering}, {family}")
ax.axhline(baseline["rmse_px"],color="black",ls=":",label="Training-mean baseline")
ax.set(xlabel="Number of POD coefficients",ylabel="Validation RMSE (sensor px)",
       title="TP position prediction: validation comparison")
ax.legend(frameon=False,fontsize=8,bbox_to_anchor=(1.02,1),loc="upper left")
save_figure(fig,"04_validation_feature_comparison")

for number,(experiment,winner) in enumerate(winners.items(),start=1):
    predicted=winner["prediction"]
    prediction_rows=[]
    for local,index in enumerate(validation):
        actual=float(y[index]); prediction=float(predicted[local])
        prediction_rows.append(dict(filename=selected[index]["filename"],
            sequence_frame_index=selected[index]["sequence_frame_index"],partition="validation",
            actual_y_px=actual,predicted_y_px=prediction,residual_px=prediction-actual,
            distance_to_boundary_px=min(abs(actual-upper),abs(actual-lower)),
            actual_category=str(categories(np.array([actual]))[0]),
            predicted_category=str(categories(np.array([prediction]))[0])))
    write_csv(OUTPUT_DIRECTORY/f"validation_predictions_{number:02d}.csv",prediction_rows)
    fig,axes=plt.subplots(1,2,figsize=(11,4),layout="constrained")
    low=min(y[validation].min(),predicted.min());high=max(y[validation].max(),predicted.max())
    axes[0].hexbin(y[validation],predicted,gridsize=35,mincnt=1,cmap="viridis")
    axes[0].plot([low,high],[low,high],"k--",lw=.8)
    axes[0].set(xlabel="Measured TP y (sensor px)",ylabel="Predicted TP y (sensor px)",
                title=f"{experiment}: validation")
    axes[0].set_aspect("equal",adjustable="box")
    axes[1].plot(time_ms[validation],y[validation],color="black",lw=.8,label="Measured")
    axes[1].plot(time_ms[validation],predicted,color="#0072B2",lw=.7,label="Predicted")
    for boundary in (upper,lower):axes[1].axhline(boundary,color="#D55E00",ls=":",lw=.8)
    axes[1].invert_yaxis()
    axes[1].set(xlabel="Time from first selected frame (ms)",ylabel="TP y (sensor px)")
    axes[1].legend(frameon=False)
    save_figure(fig,f"05_validation_prediction_{number:02d}")
    print(experiment,winner["metrics"])

metadata=dict(purpose="Training/validation TP–POD diagnostics; final test excluded",
    sources={str(p.resolve()):digest(p) for p in (coefficient_path,STATES_CSV,CATEGORY_DEFINITION,config_path)},
    n_training=len(train),n_validation=len(validation),n_regression_fit=len(fit_rows),
    mode_counts=counts,ridge_alphas=RIDGE_ALPHAS,run_nonlinear=RUN_NONLINEAR,
    svr_C=SVR_C_VALUES,svr_epsilon_px=SVR_EPSILON_PX,
    baseline=baseline,crop_studies=CROP_STUDIES if RUN_CROP_STUDY else {},
    selection="Lowest validation RMSE; no refit on validation; first grid result breaks ties.",
    uncertainty="No IID significance claims; temporally correlated samples and repeated validation selection.")
(OUTPUT_DIRECTORY/"diagnostic_manifest.json").write_text(json.dumps(metadata,indent=2))
print("Saved diagnostics:",OUTPUT_DIRECTORY)

## Interpretation and next steps

- Later modes with stronger training relationships and lower validation error support investigating a broader or training-selected feature set.
- Better RBF-SVR than Ridge performance suggests a nonlinear relationship worth investigating; it does not establish improvement for QSVM.
- Better tighter-crop results suggest that unrelated intensity variation may dilute TP information in the current POD representation.
- Weak results across these checks can reflect an inadequate representation, tracking errors, drift, or insufficient training coverage. They do not prove TP motion is absent from the images.

Inspect measurement quality and labels near boundaries. Freeze the chosen crop/features before a fresh final evaluation, preferably on another experimental record. Do not change the existing SVM/QSVM conclusions based on these validation regressions alone.